In [3]:
def tableau_to_yamanouchi(tableau):
    """
    Converts an SYT to its corresponding Yamanouchi word.
    w_j = row index (1-based) where value j is located in the tableau.
    """
    n = sum(len(row) for row in tableau)
    word = [0] * n
    for row_idx, row in enumerate(tableau):
        for val in row:
            word[val - 1] = row_idx + 1
    return tuple(word)

In [4]:
def generate_via_rsk(k, m):
    """
    Generates all balanced Yamanouchi words of shape (m^k) via RSK insertion.
    """
    n = k * m
    yamanouchi_words = set()
    
    # Iterate through all permutations of S_n
    for perm in itertools.permutations(range(1, n + 1)):
        P = []
        Q = []
        for j, val in enumerate(perm, start=1):
            r, c = rsk_insert(P, val)
            # Maintain recording tableau Q
            while len(Q) <= r:
                Q.append([])
            Q[r].append(j)
        
        # Filter for rectangular shape (m^k)
        if len(Q) == k and all(len(row) == m for row in Q):
            word = tableau_to_yamanouchi(Q)
            yamanouchi_words.add(word)
            
    return sorted(list(yamanouchi_words))

In [5]:
def promote(tableau, k, m):
    """
    Applies Schützenberger promotion to a rectangular SYT of shape (m^k).
    Uses jeu de taquin slides to slide the empty cell (1,1) to an outer boundary.
    """
    n = k * m
    # Copy tableau to avoid mutating original
    T = [list(row) for row in tableau]
    
    r, c = 0, 0  # Start empty cell at top-left
    T[0][0] = None
    
    # Jeu de taquin slides
    while True:
        right_val = T[r][c + 1] if c + 1 < m else float('inf')
        down_val = T[r + 1][c] if r + 1 < k else float('inf')
        
        if right_val == float('inf') and down_val == float('inf'):
            break  # Reached bottom-right corner boundary
            
        if right_val < down_val:
            T[r][c] = right_val
            c += 1
        else:
            T[r][c] = down_val
            r += 1
            
    # Shift values down by 1 and place n in the final empty cell
    for r_idx in range(k):
        for c_idx in range(m):
            if T[r_idx][c_idx] is not None:
                T[r_idx][c_idx] -= 1
                
    T[r][c] = n
    return T


# --- Example Execution ---
if __name__ == "__main__":
    k, m = 3, 2  # 3 rows, 2 columns (alphabet {1, 2, 3}, m = 2)
    
    print(f"=== Generating Yamanouchi Words (k={k}, m={m}) via RSK ===")
    words = generate_via_rsk(k, m)
    for idx, w in enumerate(words, start=1):
        print(f"Word {idx}: {' '.join(map(str, w))}")
    print(f"Total count: {len(words)}")

    print("\n=== Demonstrating Promotion Orbit ===")
    # Initial tableau for shape 3x2
    initial_tableau = [
        [1, 2],
        [3, 5],
        [4, 6]
    ]
    
    curr = initial_tableau
    for step in range(k * m):
        w = tableau_to_yamanouchi(curr)
        print(f"Step {step}: Tableau {curr} -> Yamanouchi Word: {' '.join(map(str, w))}")
        curr = promote(curr, k, m)


=== Generating Yamanouchi Words (k=3, m=2) via RSK ===
Word 1: 1 1 2 2 3 3
Word 2: 1 1 2 3 2 3
Word 3: 1 2 1 2 3 3
Word 4: 1 2 1 3 2 3
Word 5: 1 2 3 1 2 3
Total count: 5

=== Demonstrating Promotion Orbit ===
Step 0: Tableau [[1, 2], [3, 5], [4, 6]] -> Yamanouchi Word: 1 1 2 3 2 3
Step 1: Tableau [[1, 4], [2, 5], [3, 6]] -> Yamanouchi Word: 1 2 3 1 2 3
Step 2: Tableau [[1, 3], [2, 4], [5, 6]] -> Yamanouchi Word: 1 2 1 2 3 3
Step 3: Tableau [[1, 2], [3, 5], [4, 6]] -> Yamanouchi Word: 1 1 2 3 2 3
Step 4: Tableau [[1, 4], [2, 5], [3, 6]] -> Yamanouchi Word: 1 2 3 1 2 3
Step 5: Tableau [[1, 3], [2, 4], [5, 6]] -> Yamanouchi Word: 1 2 1 2 3 3


In [6]:
def generate_balanced_yamanouchi(k: int, m: int):
    """
    Directly generates all balanced Yamanouchi words of shape (m^k)
    using backtracking with instant branch pruning.
    
    Parameters:
        k (int): Number of distinct symbols / rows (alphabet size {1, ..., k})
        m (int): Frequency of each symbol / row length (shape m^k)
        
    Yields:
        tuple: A valid balanced Yamanouchi word of length k * m.
    """
    total_len = k * m
    counts = [0] * k
    current_word = []

    def backtrack():
        if len(current_word) == total_len:
            yield tuple(current_word)
            return

        for symbol in range(1, k + 1):
            idx = symbol - 1  # 0-indexed for counts
            
            # Rule 1: Capacity check - symbol cannot exceed m occurrences
            if counts[idx] >= m:
                continue
                
            # Rule 2: Yamanouchi check - symbol x needs count(x-1) > count(x)
            if symbol > 1 and counts[idx - 1] <= counts[idx]:
                continue

            # Choose
            counts[idx] += 1
            current_word.append(symbol)

            # Explore
            yield from backtrack()

            # Backtrack
            current_word.pop()
            counts[idx] -= 1

    yield from backtrack()


In [7]:
def total_words (k: int, m: int):
    #takes the dimentions of the tableaux and uses the Hook Length Formula to return the number of words

SyntaxError: incomplete input (1153719847.py, line 2)

In [8]:
# --- Execution & Benchmark Comparison ---
if __name__ == "__main__":
    import time
    
    k, m = 3, 3  # Shape (k^m), k letters and m instances of those letters
    
    start_time = time.perf_counter()
    words = list(generate_balanced_yamanouchi(k, m)) #an array of all the possible words
    total_possible_combos = int(math.factorial(len(words[0]))/numpy.array(words[0]).prod()) #The total number of words
    #given k and m given by the Hook length formula (there are duplicates)
    elapsed = time.perf_counter() - start_time
    
    print(f"=== Backtracking Generator (k={k}, m={m}) ===")
    print(f"Total Words Generated: {len(words)}")
    print(f"Execution Time: {elapsed:.6f} seconds\n")
    print(f"Total possible combinations: {total_possible_combos}")
    print(f"Mul of the array: {numpy.prod(words[0])}")
    print(f"Number of letters: {len(words[0])}")
    
    print("Generated Words:")
    for w in words[:total_possible_combos]:
        print(" ".join(map(str, w)))

=== Backtracking Generator (k=3, m=3) ===
Total Words Generated: 42
Execution Time: 0.000376 seconds

Total possible combinations: 1680
Mul of the array: 216
Number of letters: 9
Generated Words:
1 1 1 2 2 2 3 3 3
1 1 1 2 2 3 2 3 3
1 1 1 2 2 3 3 2 3
1 1 1 2 3 2 2 3 3
1 1 1 2 3 2 3 2 3
1 1 2 1 2 2 3 3 3
1 1 2 1 2 3 2 3 3
1 1 2 1 2 3 3 2 3
1 1 2 1 3 2 2 3 3
1 1 2 1 3 2 3 2 3
1 1 2 2 1 2 3 3 3
1 1 2 2 1 3 2 3 3
1 1 2 2 1 3 3 2 3
1 1 2 2 3 1 2 3 3
1 1 2 2 3 1 3 2 3
1 1 2 2 3 3 1 2 3
1 1 2 3 1 2 2 3 3
1 1 2 3 1 2 3 2 3
1 1 2 3 2 1 2 3 3
1 1 2 3 2 1 3 2 3
1 1 2 3 2 3 1 2 3
1 2 1 1 2 2 3 3 3
1 2 1 1 2 3 2 3 3
1 2 1 1 2 3 3 2 3
1 2 1 1 3 2 2 3 3
1 2 1 1 3 2 3 2 3
1 2 1 2 1 2 3 3 3
1 2 1 2 1 3 2 3 3
1 2 1 2 1 3 3 2 3
1 2 1 2 3 1 2 3 3
1 2 1 2 3 1 3 2 3
1 2 1 2 3 3 1 2 3
1 2 1 3 1 2 2 3 3
1 2 1 3 1 2 3 2 3
1 2 1 3 2 1 2 3 3
1 2 1 3 2 1 3 2 3
1 2 1 3 2 3 1 2 3
1 2 3 1 1 2 2 3 3
1 2 3 1 1 2 3 2 3
1 2 3 1 2 1 2 3 3
1 2 3 1 2 1 3 2 3
1 2 3 1 2 3 1 2 3
